# Análise Exploratória das Tribos Literárias (Data Storytelling)

Neste notebook, faremos uma jornada exploratória para desvendar a personalidade e os hábitos de leitura das **4 Tribos Literárias** que mapeamos em nosso catálogo de livros do Booklog. 

Em vez de olharmos apenas para números isolados, buscaremos contar a história de quem são essas pessoas: o que elas leem, como elas avaliam suas obras, se preferem livros rápidos ou calhamaços densos, e como essas características as agrupam em "galáxias" distintas de comportamento.

### As Nossas Quatro Personagens (As Tribos):
* **Tribo 0: "Universo Geek e Fantasia Pop"** (Fãs de mundos fantásticos, magia e sagas de ficção científica. Grupo de maior engajamento viral da plataforma).
* **Tribo 1: "Literatura Sênior, Ensaios e Biografias"** (Leitores exigentes que leem biografias densas, história e obras híbridas literárias/filosóficas).
* **Tribo 2: "Não-Ficção de Nicho e Estilo de Vida"** (Profissionais e curiosos buscando desenvolvimento prático, carreiras, guias e estilo de vida).
* **Tribo 3: "Romances Mainstream e Dramas"** (Devoradores de romances, dramas urbanos e histórias de amor e entretenimento).


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
from scipy.stats import kruskal

print("Bibliotecas importadas com sucesso!")


Bibliotecas importadas com sucesso!


## 1. Entrando na Biblioteca: Carregamento do Catálogo com Clusters
Carregamos os dados contendo a designação de cada livro para a sua respectiva Tribo Literária (Cluster).

> **Nota sobre o número de colunas (De 10 para 17 colunas):**
> O dataset original (`books.parquet`) possuía 10 colunas brutas. No entanto, o nosso dataset processado final agora conta com **17 colunas**. Por que isso aconteceu?
> 1. **Engenharia de Gêneros Pivotados (9 colunas):** Em vez de mantermos uma única coluna com listas de gêneros em formato texto, transformamos os gêneros consolidados em **9 colunas binárias** (0 ou 1) independentes.
> 2. **Variáveis Numéricas Originais (3 colunas):** Mantivemos `rating`, `pages` e `totalratings`.
> 3. **Identidade e Metadados do Modelo (5 colunas):** Temos as colunas `title` e `author` (identificação), a coluna `Cluster` (grupo classificado pelo K-Means) e as coordenadas de projeção visual `svd_x` e `svd_y`.
>
> Essa transformação (12 variáveis de modelagem + 5 de suporte) é o que viabiliza a interpretabilidade dos clusters e a plotagem do dashboard.


In [2]:
# Carregar os dados de livros com clusters
df = pd.read_parquet('Machine Learning/data/processed/livros_com_clusters.parquet')

# Função para corrigir codificação dupla de caracteres UTF-8 (ex: 'Charlotte BrontÃ«' -> 'Charlotte Brontë')
def clean_double_encoding(s):
    if not isinstance(s, str):
        return s
    try:
        b = s.encode('latin-1')
        decoded = b.decode('utf-8')
        if len(decoded) < len(s):
            return decoded
        return s
    except Exception:
        return s

df['author'] = df['author'].apply(clean_double_encoding)
df['title'] = df['title'].apply(clean_double_encoding)

print(f"Catálogo carregado com {df.shape[0]:,} livros e {df.shape[1]} colunas (com codificação limpa).")
print("\nExemplo dos primeiros livros com suas colunas e coordenadas:")
print(df[['title', 'author', 'Cluster', 'svd_x', 'svd_y']].head(3))


Catálogo carregado com 81,979 livros e 17 colunas (com codificação limpa).
Exemplo dos primeiros livros com suas colunas e coordenadas:
                                    title  ...     svd_y
0                              "Daisuki."  ... -0.127454
1       "Dark Pictures" and Other Stories  ...  0.222201
2  "Defects": Engendering the Modern Body  ...  0.167648
[3 rows x 5 columns]


## 2. Capítulo 1: O Tamanho das Tribos
A primeira pergunta da nossa história é: **como o catálogo do Booklog está distribuído?** Será que temos uma tribo gigantesca e outras minúsculas, ou as três dividem o catálogo de forma equilibrada?

Plotamos o volume de livros em cada tribo para entender essa divisão.


In [3]:
# Contagem e percentual por cluster
dist_df = df['Cluster'].value_counts().reset_index()
dist_df.columns = ['Tribo', 'Quantidade']
dist_df['Percentual'] = (dist_df['Quantidade'] / dist_df['Quantidade'].sum()) * 100
dist_df['Nome_Curto'] = dist_df['Tribo'].map({
    0: 'Tribo 0',
    1: 'Tribo 1',
    2: 'Tribo 2',
    3: 'Tribo 3'
})
dist_df['Nome_Longo'] = dist_df['Tribo'].map({
    0: 'Tribo 0: Universo Geek & Fantasia',
    1: 'Tribo 1: Literatura Sênior & Ensaios',
    2: 'Tribo 2: Não-Ficção de Nicho & Lazer',
    3: 'Tribo 3: Romances Mainstream & Dramas'
})

# Mapear cores solicitadas: Tribo 0 = Vermelho, Tribo 1 = Azul, Tribo 2 = Verde, Tribo 3 = Laranja
CORES_MAP = {
    'Tribo 0: Universo Geek & Fantasia': '#e41a1c', # Vermelho
    'Tribo 1: Literatura Sênior & Ensaios': '#377eb8', # Azul
    'Tribo 2: Não-Ficção de Nicho & Lazer': '#4daf4a', # Verde
    'Tribo 3: Romances Mainstream & Dramas': '#ff7f00'  # Laranja
}

# Criar gráfico de barras com nomes curtos no X e a legenda explicativa na lateral
fig_dist = px.bar(
    dist_df,
    x='Nome_Curto',
    y='Quantidade',
    text=dist_df['Percentual'].apply(lambda x: f"{x:.1f}%"),
    title='Distribuição dos Livros do Catálogo pelas 4 Tribos Literárias',
    color='Nome_Longo',
    color_discrete_map=CORES_MAP,
    category_orders={'Nome_Curto': ['Tribo 0', 'Tribo 1', 'Tribo 2', 'Tribo 3']},
    labels={'Quantidade': 'Número de Livros', 'Nome_Curto': 'Tribo Literária', 'Nome_Longo': 'Legenda (Descrição)'}
)
fig_dist.update_traces(textposition='outside', cliponaxis=False)
fig_dist.update_yaxes(range=[0, dist_df['Quantidade'].max() * 1.15])
fig_dist.update_layout(
    height=450, 
    showlegend=True, 
    legend=dict(title_text='Legenda das Tribos', yanchor="top", y=1, xanchor="left", x=1.02),
    margin=dict(t=50, b=20, l=20, r=150)
)
fig_dist.show()


* **Insight do Storytelling:** O catálogo mostra uma divisão equilibrada e muito rica sob a perspectiva de 4 clusters:
  * A **Tribo 2 (Não-Ficção de Nicho & Lazer)** é o maior grupo (35.6%), reunindo livros voltados a desenvolvimento profissional, manuais e estilo de vida.
  * A **Tribo 1 (Literatura Sênior & Ensaios)** representa 23.9%, englobando leitores de biografias, livros históricos e ensaios profundos.
  * A **Tribo 3 (Romances Mainstream & Dramas)** responde por 22.0% dos livros, focando na ficção popular de romance e drama comercial.
  * A **Tribo 0 (Universo Geek & Fantasia)** representa 18.5%, concentrando a maior parte da ficção científica e fantasia pop.

---

## 3. Capítulo 2: A Preferência Literária das Tribos
Agora, vamos investigar o **gosto literário de cada grupo**. O que define a identidade de leitura dessas tribos? 

Para isso, calculamos a taxa de presença de cada um dos 9 gêneros principais dentro de cada cluster, nos dando o "DNA" de cada grupo.

> **Importante: Por que a soma das barras de uma mesma tribo ultrapassa 100%?**
> Ao olhar para o gráfico a seguir, você notará que a soma das porcentagens dos gêneros de um mesmo cluster ultrapassa 100%. **Isso não é um erro!**
> Isso acontece porque **um mesmo livro pode ter múltiplos gêneros marcados simultaneamente (dados multilabel)**. Por exemplo, uma mesma obra pode ser marcada como "Romance" e também "Ficção Geral e Literatura".
> A barra no gráfico não representa uma divisão exclusiva do catálogo, mas sim: *"Qual a porcentagem de livros desta tribo que possui este gênero marcado?"*. Como os livros se sobrepõem entre os gêneros, as categorias não são excludentes, fazendo com que a soma natural dos percentuais no cluster seja maior do que 100%.


In [4]:
genre_cols = [
    'Artes, Lazer e Estilo de Vida',
    'Fantasia e Ficção Científica',
    'Ficção Geral e Literatura',
    'História e Biografia',
    'Infantojuvenil e Quadrinhos',
    'Mistério, Thriller e Terror',
    'Não-Ficção e Autodesenvolvimento',
    'Outros',
    'Romance'
]

# Calcular a proporção de cada gênero por cluster
proporcoes = df.groupby('Cluster')[genre_cols].mean().reset_index()

# Pivotar dados para o formato longo (long format) para exibição no Plotly
prop_melted = proporcoes.melt(id_vars='Cluster', var_name='Gênero', value_name='Proporção')
prop_melted['Percentual'] = prop_melted['Proporção'] * 100
prop_melted['Nome_Tribo'] = prop_melted['Cluster'].map({
    0: 'Tribo 0 (Universo Geek)',
    1: 'Tribo 1 (Literatura Sênior)',
    2: 'Tribo 2 (Não-Ficção & Lazer)',
    3: 'Tribo 3 (Romance & Drama)'
})

# Mapear cores solicitadas: Tribo 0 = Vermelho, Tribo 1 = Azul, Tribo 2 = Verde, Tribo 3 = Laranja
CORES_GENRES = {
    'Tribo 0 (Universo Geek)': '#e41a1c', # Vermelho
    'Tribo 1 (Literatura Sênior)': '#377eb8', # Azul
    'Tribo 2 (Não-Ficção & Lazer)': '#4daf4a', # Verde
    'Tribo 3 (Romance & Drama)': '#ff7f00'  # Laranja
}

# Criar gráfico de barras agrupado por Tribo
fig_genres = px.bar(
    prop_melted,
    x='Percentual',
    y='Gênero',
    color='Nome_Tribo',
    barmode='group',
    title='Presença de Gêneros Literários por Tribo',
    color_discrete_map=CORES_GENRES,
    labels={'Percentual': 'Presença no Cluster (%)', 'Gênero': 'Gênero Literário', 'Nome_Tribo': 'Tribo'}
)
fig_genres.update_layout(height=650, yaxis={'categoryorder':'total ascending'}, margin=dict(t=50, b=20, l=20, r=20))
fig_genres.show()


* **Análise Detalhada do DNA Literário (Storytelling):**
  * **Tribo 0 (Vermelho - Universo Geek):** Focado 100% em *Fantasia e Ficção Científica*, com forte presença de *Ficção Geral e Literatura* (87%) e *Infantojuvenil e Quadrinhos* (46%). Representa a comunidade de leitores de sagas e mundos fantásticos.
  * **Tribo 1 (Azul - Literatura Sênior & Ensaios):** Esta tribo agrupa obras com as tags de *Ficção Geral e Literatura* (100% de presença) e *Não-Ficção e Autodesenvolvimento* (100%). Possui também alta taxa de *História e Biografia* (56%). Representa biografias literárias, ensaios filosóficos e clássicos intelectuais que mesclam literatura e realidade.
  * **Tribo 2 (Verde - Não-Ficção & Lazer):** Reúne obras práticas e informativas, com *Não-Ficção* (88% de presença) e *Outros* (75%), além de *Artes, Lazer e Estilo de Vida* (37%). É o grupo de manuais, guias profissionais e livros de hobbies.
  * **Tribo 3 (Laranja - Romance & Drama):** Reúne livros com *Romance* (46% de presença) e *Ficção Geral* (77%), sem nenhuma presença de Fantasia (0%). Representa a literatura romântica popular e dramas de entretenimento.

> **Por que a Tribo 1 tem 100% de Não-Ficção e 100% de Literatura ao mesmo tempo?**
> Isso reflete o fato de que muitas obras de história, memórias narrativas, biografias intelectuais ou ensaios filosóficos são classificadas tanto como "Não-Ficção" quanto como "Literatura". O K-Means identificou essa interseção e a isolou no Cluster 1, agrupando os livros com maior densidade intelectual e textual.

---

## 4. Capítulo 3: Os Hábitos de Consumo e Avaliação
Sabemos o que eles leem, mas **como** eles consomem esses livros?
* Eles leem calhamaços ou livros de leitura rápida?
* Suas tribos são rigorosas ou generosas nas notas médias?
* Suas leituras são fenômenos mundiais (milhões de avaliações) ou obras de nicho?

Analisamos isso através de boxplots comparativos para a Nota Média (`rating`), Extensão (`pages`) e Popularidade (`totalratings`).


In [5]:
fig_box = make_subplots(
    rows=1, cols=3, 
    subplot_titles=('Nota Média (Rating)', 'Extensão (Páginas)', 'Popularidade (Avaliações)')
)

# Copiar dataframe e mapear nomes legíveis
df_plot = df.copy()
df_plot['Nome_Tribo'] = df_plot['Cluster'].map({
    0: 'Tribo 0 (Universo Geek)',
    1: 'Tribo 1 (Literatura Sênior)',
    2: 'Tribo 2 (Não-Ficção & Lazer)',
    3: 'Tribo 3 (Romance & Drama)'
})

# Adicionar Boxplots individuais para cada cluster com cores estritas
for cluster_id, color in zip([0, 1, 2, 3], ['#e41a1c', '#377eb8', '#4daf4a', '#ff7f00']):
    subset = df_plot[df_plot['Cluster'] == cluster_id]
    
    fig_box.add_trace(
        go.Box(y=subset['rating'], name=f"Tribo {cluster_id}", marker_color=color, showlegend=False),
        row=1, col=1
    )
    fig_box.add_trace(
        go.Box(y=subset['pages'], name=f"Tribo {cluster_id}", marker_color=color, showlegend=False),
        row=1, col=2
    )
    fig_box.add_trace(
        go.Box(y=subset['totalratings'] + 1, name=f"Tribo {cluster_id}", marker_color=color, showlegend=False),
        row=1, col=3
    )

# Definir limites de zoom para podermos ler a caixa central ignorando os outliers gigantescos
fig_box.update_yaxes(range=[2.5, 5.0], row=1, col=1)
fig_box.update_yaxes(range=[0, 650], row=1, col=2)
fig_box.update_yaxes(type="log", row=1, col=3) # Escala logarítmica para lidar com a enorme disparidade de popularidade

fig_box.update_layout(
    title_text='Distribuição das Características Físicas e de Popularidade por Tribo (Escala Log em Avaliações)',
    height=480,
    margin=dict(t=80, b=20, l=20, r=20)
)
fig_box.show()


* **Insights de Storytelling dos Hábitos:**
  * **As Notas:** A **Tribo 2 (Não-Ficção de Nicho)** e a **Tribo 1 (Literatura Sênior)** têm avaliações médias excelentes (mediana de ~3.94 e 3.90). A **Tribo 3 (Romance & Drama)** tem a menor nota mediana (3.81), mostrando que o público de romances e ficção comercial é extremamente crítico e ativo.
  * **O Tamanho:** A **Tribo 1 (Literatura Sênior)** lê os livros mais longos (média próxima a 290 páginas), seguidos de perto pela **Tribo 2 (Não-Ficção)** com 280 páginas. A **Tribo 3 (Romance & Drama)** tem os livros mais curtos em média (231 páginas), confirmando que a ficção comercial preza por leituras rápidas e fluidas.
  * **A Popularidade (Avaliações - Escala Logarítmica):** Ao aplicarmos a escala logarítmica (ajustando +1), conseguimos visualizar claramente a disparidade: a **Tribo 0 (Universo Geek)** é o maior fenômeno de engajamento da plataforma, com mediana muito superior e média impressionante de **7.973 avaliações por livro**, seguida pela **Tribo 3 (Romance & Drama)** com 3.904. A **Tribo 2 (Não-Ficção de Nicho)** é claramente composta por livros de menor apelo geral, com mediana muito baixa (nicho). A escala logarítmica salvou o gráfico de ser 'esmagado' pelas obras com milhões de avaliações.


---

## 4. Capítulo 4: Quem Lidera a Tribo? Os Autores de Maior Engajamento
Agora que entendemos a popularidade geral, quem são as **figuras públicas de destaque** dentro de cada Tribo Literária?
Calculamos a soma das avaliações recebidas por autor dentro de cada cluster e selecionamos o Top 5. Isso nos revela os nomes emblemáticos que definem o ecossistema de cada comunidade de leitores.


In [6]:
# Subplots: 2x2 para o Top 5 Autores de cada Tribo
fig_authors = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Tribo 0 (Geek)',
        'Tribo 1 (Literatura Sênior)',
        'Tribo 2 (Não-Ficção)',
        'Tribo 3 (Romance & Drama)'
    ),
    horizontal_spacing=0.28,  # Aumentado para evitar que as barras e rótulos colidam
    vertical_spacing=0.22     # Aumentado para evitar que títulos de subplots se sobreponham
)

clusters = [0, 1, 2, 3]
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#ff7f00']

for i, (cluster_id, color) in enumerate(zip(clusters, colors)):
    row = (i // 2) + 1
    col = (i % 2) + 1
    
    # Filtrar cluster e agrupar por autor
    top_authors = df[df['Cluster'] == cluster_id].groupby('author')['totalratings'].sum().reset_index()
    top_authors = top_authors.sort_values(by='totalratings', ascending=True).tail(5)
    
    # Encurtar nomes de autores gigantes para caber no gráfico
    top_authors['author_short'] = top_authors['author'].apply(lambda x: x.split(',')[0][:18])
    
    fig_authors.add_trace(
        go.Bar(
            x=top_authors['totalratings'],
            y=top_authors['author_short'],
            orientation='h',
            marker_color=color,
            name=f"Tribo {cluster_id}",
            showlegend=False
        ),
        row=row, col=col
    )

fig_authors.update_layout(
    title_text='Autores Líderes de Engajamento por Tribo Literária (Soma de Avaliações)',
    height=600,
    width=1000,               # Definido largura maior para dar espaçamento de layout
    margin=dict(t=80, b=40, l=120, r=20) # Margem esquerda aumentada para dar espaço aos rótulos de autores
)
fig_authors.show()


* **Insights de Storytelling dos Líderes:**
  * **Tribo 0 (Geek):** Liderada por gigantes do imaginário popular, como **Stephen King**, **Veronica Roth**, **George R.R. Martin** e **J.K. Rowling**. São autores de sagas de fantasia e mistério que carregam milhões de leitores fiéis.
  * **Tribo 1 (Literatura Sênior):** Nomes clássicos e de peso literário como **Charlotte Brontë**, **Homer**, **Elie Wiesel** e **William Shakespeare**. Esse grupo representa leitores voltados a clássicos históricos e literários consolidados.
  * **Tribo 2 (Não-Ficção de Nicho):** Líderes como **Walter Isaacson** (famoso por biografias de Steve Jobs e Einstein), **Steven D. Levitt** (Freakonomics), **Charles Duhigg** (O Poder do Hábito) e **Daniel Kahneman** (Rápido e Devagar). São livros de negócios, ciência aplicada, desenvolvimento e biografias práticas.
  * **Tribo 3 (Romance & Drama):** Liderada por autores do romance mainstream e drama urbano contemporâneo, como **John Green**, **E.L. James** (Cinquenta Tons de Cinza), **Agatha Christie**, **Jodi Picoult** e **Rainbow Rowell**.

---

## 5. Capítulo 5: O Efeito Calhamaço - Extensão vs. Nota Média
Uma das discussões mais interessantes no mundo da leitura é: **será que livros maiores recebem notas melhores?**
Dividimos as obras em 4 faixas de extensão: *Curto* (<150 páginas), *Médio* (150 a 350 páginas), *Longo* (350 a 600 páginas) e *Calhamaço* (>600 páginas). Em seguida, calculamos a média de avaliação (rating) para cada faixa dentro de cada uma das 4 tribos.


In [7]:
# Preparar dados categorizados por faixa de páginas
df_size = df.copy()
df_size['Tamanho'] = pd.cut(
    df_size['pages'], 
    bins=[-1, 150, 350, 600, 99999], 
    labels=['Curto (<150 págs)', 'Médio (150-350 págs)', 'Longo (350-600 págs)', 'Calhamaço (>600 págs)']
)

df_size_grouped = df_size.groupby(['Cluster', 'Tamanho'])['rating'].mean().reset_index()
df_size_grouped['Nome_Tribo'] = df_size_grouped['Cluster'].map({
    0: 'Tribo 0 (Universo Geek)',
    1: 'Tribo 1 (Literatura Sênior)',
    2: 'Tribo 2 (Não-Ficção & Lazer)',
    3: 'Tribo 3 (Romance & Drama)'
})

# Criar gráfico de barras agrupadas
fig_size_rating = px.bar(
    df_size_grouped,
    x='Tamanho',
    y='rating',
    color='Nome_Tribo',
    barmode='group',
    title='O Efeito Calhamaço: Relação entre Extensão e Nota Média por Tribo',
    color_discrete_map={
        'Tribo 0 (Universo Geek)': '#e41a1c',
        'Tribo 1 (Literatura Sênior)': '#377eb8',
        'Tribo 2 (Não-Ficção & Lazer)': '#4daf4a',
        'Tribo 3 (Romance & Drama)': '#ff7f00'
    },
    labels={'rating': 'Nota Média', 'Tamanho': 'Extensão do Livro', 'Nome_Tribo': 'Tribo'}
)

# Ajustar zoom vertical para focar na diferença de nota (entre 3.5 e 4.2)
fig_size_rating.update_yaxes(range=[3.5, 4.25])
fig_size_rating.update_layout(
    height=480, 
    legend=dict(title_text='Tribo Literária', yanchor="top", y=1, xanchor="left", x=1.02),
    margin=dict(t=50, b=20, l=20, r=180)
)
fig_size_rating.show()


* **O Efeito de Auto-Seleção e Engajamento:**
  O gráfico revela um comportamento padrão incrível em **todas as quatro tribos**: à medida que a extensão do livro aumenta, a nota média também sobe, atingindo o pico nos **Calhamaços (>600 páginas)**.
  
  Existem duas explicações fortes para isso:
  1. **Filtro de Auto-Seleção (Compromisso):** Um leitor que decide iniciar um livro de mais de 600 páginas geralmente já é muito fã do autor, da série ou do tema. Portanto, ele tem uma probabilidade muito menor de abandonar ou avaliar negativamente a obra do que alguém que lê um livro curto de forma casual.
  2. **Imersão Narrativa/Conteúdo:** Obras mais longas permitem maior desenvolvimento de personagens, construção de mundos densos (no caso da fantasia) ou maior profundidade técnica (na não-ficção), gerando maior satisfação e conexão com o leitor.
  
  A maior disparidade ocorre na **Tribo 3 (Romance & Drama)**, onde livros curtos têm média de **3.76** e calhamaços sobem para **4.11** (+0.35 pontos).

---

## 6. Capítulo 6: Validação Científica da Diferença de Perfis
Para provar que essas diferenças de comportamento (páginas, avaliações e notas) não aconteceram por acaso, aplicamos o teste estatístico **Kruskal-Wallis**. Este teste analisa se as distribuições das 4 tribos são realmente diferentes de forma estatisticamente significativa.


In [8]:
print("=== Validação Estatística de Diferenças entre Grupos (Kruskal-Wallis) ===")
print("Hipótese Nula (H0): Não há diferença nas distribuições das variáveis entre os clusters.")
print("Hipótese Alternativa (H1): Pelo menos um cluster apresenta distribuição diferente.")
print("-" * 75)

for col in ['rating', 'pages', 'totalratings']:
    g0 = df[df['Cluster'] == 0][col]
    g1 = df[df['Cluster'] == 1][col]
    g2 = df[df['Cluster'] == 2][col]
    g3 = df[df['Cluster'] == 3][col]
    
    stat, p_val = kruskal(g0, g1, g2, g3)
    print(f"Variável: {col:<12} | Estatística H: {stat:10.2f} | p-value: {p_val:.2e}")
    if p_val < 0.05:
        print(f"  -> Conclusão: Rejeitamos H0. A diferença entre as tribos é ESTATISTICAMENTE SIGNIFICATIVA (p < 0.05).")
    else:
        print(f"  -> Conclusão: Aceitamos H0. Não há diferença estatisticamente significativa.")
    print("-" * 75)


=== Validação Estatística de Diferenças entre Grupos (Kruskal-Wallis) ===
Hipótese Nula (H0): Não há diferença nas distribuições das variáveis entre os clusters.
Hipótese Alternativa (H1): Pelo menos um cluster apresenta distribuição diferente.
---------------------------------------------------------------------------
Variável: rating       | Estatística H:    1667.74 | p-value: 0.00e+00
  -> Conclusão: Rejeitamos H0. A diferença entre as tribos é ESTATISTICAMENTE SIGNIFICATIVA (p < 0.05).
---------------------------------------------------------------------------
Variável: pages        | Estatística H:    1210.64 | p-value: 3.61e-262
  -> Conclusão: Rejeitamos H0. A diferença entre as tribos é ESTATISTICAMENTE SIGNIFICATIVA (p < 0.05).
---------------------------------------------------------------------------
Variável: totalratings | Estatística H:   17960.99 | p-value: 0.00e+00
  -> Conclusão: Rejeitamos H0. A diferença entre as tribos é ESTATISTICAMENTE SIGNIFICATIVA (p < 0.05).
-

* **Insight Acadêmico:** Com um p-value de praticamente zero ($p < 0.001$), rejeitamos a hipótese nula com segurança máxima. Isso prova que o algoritmo K-Means separou o catálogo em grupos que possuem hábitos de leitura **inerentemente diferentes** em termos de nota, tamanho de livros e popularidade.

---

## 7. Capítulo 7: Mapeamento da Galáxia do Booklog (Mapa 2D do SVD)
Finalmente, vamos olhar para o **mapa espacial** dessas tribos. 
Usamos a projeção de **SVD (Singular Value Decomposition)** nas 12 variáveis de modelagem para plotar uma amostra de 10.000 livros em duas dimensões.

Cada ponto representa um livro, e a distância espacial reflete a semelhança entre eles.


In [9]:
# Amostra de 10.000 livros para performance visual no navegador
df_sample = df.sample(10000, random_state=42).copy()
df_sample['Cluster_Nome'] = df_sample['Cluster'].map({
    0: 'Tribo 0 (Universo Geek & Fantasia)',
    1: 'Tribo 1 (Literatura Sênior & Ensaios)',
    2: 'Tribo 2 (Não-Ficção de Nicho & Lazer)',
    3: 'Tribo 3 (Romances Mainstream & Dramas)'
})

# Mapear cores solicitadas: Tribo 0 = Vermelho, Tribo 1 = Azul, Tribo 2 = Verde, Tribo 3 = Laranja
CORES_SCATTER = {
    'Tribo 0 (Universo Geek & Fantasia)': '#e41a1c', # Vermelho
    'Tribo 1 (Literatura Sênior & Ensaios)': '#377eb8', # Azul
    'Tribo 2 (Não-Ficção de Nicho & Lazer)': '#4daf4a', # Verde
    'Tribo 3 (Romances Mainstream & Dramas)': '#ff7f00'  # Laranja
}

# Gerar gráfico de dispersão 2D com as coordenadas SVD originais
fig_scatter = px.scatter(
    df_sample,
    x='svd_x',
    y='svd_y',
    color='Cluster_Nome',
    hover_name='title',
    hover_data=['author', 'rating', 'pages'],
    title='Mapa Espacial da Galáxia de Livros (Amostra de 10.000 títulos via SVD)',
    category_orders={'Cluster_Nome': [
        'Tribo 0 (Universo Geek & Fantasia)',
        'Tribo 1 (Literatura Sênior & Ensaios)',
        'Tribo 2 (Não-Ficção de Nicho & Lazer)',
        'Tribo 3 (Romances Mainstream & Dramas)'
    ]},
    color_discrete_map=CORES_SCATTER,
    labels={'svd_x': 'Componente de Projeção SVD 1', 'svd_y': 'Componente de Projeção SVD 2'}
)

fig_scatter.update_traces(marker=dict(size=4, opacity=0.6, line=dict(width=0.2, color='DarkSlateGrey')))
fig_scatter.update_layout(
    height=600,
    showlegend=True,
    legend=dict(
        title_text='Tribos Literárias',
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02
    ),
    margin=dict(t=80, b=20, l=20, r=180) # give margin for legend on the right
)
fig_scatter.show()


## Conclusão da Jornada: Resumo das Tribos

Ao fim do nosso Storytelling, podemos categorizar as 4 personas de leitura do Booklog:

1. **Os Geeks e Fãs de Fantasia (Tribo 0):** Representam 18.5% do catálogo. São leitores vorazes, extremamente vocais e apaixonados (líderes absolutos de popularidade de resenhas). Seus livros contam com alta presença de magia, aventura e ficção científica.
2. **Os Intelectuais de Clássicos e Biografias (Tribo 1):** Representam 23.9% do catálogo. Lêm biografias, ensaios e clássicos literários longos de alta densidade intelectual.
3. **Os Práticos e Especialistas (Tribo 2):** Representam 35.6% do catálogo. Leem livros práticos sobre carreiras, autodesenvolvimento e lazer, com leituras muito focadas e notas altas, mantendo-se em um perfil de menor exposição geral (nichos).
4. **Os Românticos e Fãs de Drama (Tribo 3):** Representam 22.0% do catálogo. Devoram romances e dramas comerciais, preferindo leituras mais rápidas e fluidas de alta identificação comunitária.

Essas informações são valiosas para orientar recomendações hiper-direcionadas de catálogo e refinar a experiência do usuário do Booklog!
